# Kurtz Convergence Theorem

Kurtz's theorem: convergence of scaled sample paths to the mean-field limit.

For volume parameter N, the stochastic propensities are:
    a0 = k0 * N                       (zero-order, scales with volume)
    a1 = (k1 / N) * X(X-1)/2          (bimolecular, inversely scales)
    a2 = k2 * X                       (first-order, no scaling)
    a3 = k3 * Y

The rescaled process  x^N(t) = X(t)/N  converges in probability
to the deterministic RRE solution  x(t)  as  N -> infinity.
Fluctuations around x(t) are of order 1/sqrt(N).

**Figures:** kurtz_convergence.svg

In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams.update({
    "svg.fonttype": "path", "mathtext.fontset": "cm",
    "font.family": "serif", "font.size": 11,
    "axes.labelsize": 13, "axes.titlesize": 13,
    "legend.fontsize": 9, "xtick.labelsize": 10, "ytick.labelsize": 10,
    "lines.linewidth": 1.4, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
})

# ── Rate constants (macroscopic / concentration units) ────────
k0 = 10.0       # production      (conc / time)
k1 = 0.005      # dimerisation    (1 / conc·time)
k2 = 0.1        # X degradation   (1 / time)
k3 = 0.1        # Y degradation   (1 / time)

T    = 40.0
seed = 42

# ── Deterministic RRE (mean-field limit) ──────────────────────
def rre(t, c):
    x, y = c
    return [k0 - k1*x**2 - k2*x,
            0.5*k1*x**2 - k3*y]

sol = solve_ivp(rre, [0, T], [0.0, 0.0],
                dense_output=True, max_step=0.1, rtol=1e-8)
tg = np.linspace(0, T, 1200)
x_det, y_det = sol.sol(tg)
print(f"RRE steady state: x*={x_det[-1]:.2f}, y*={y_det[-1]:.2f}")

# ── SSA with volume scaling ───────────────────────────────────
def gillespie_N(T, N, sd):
    """Gillespie SSA at volume N. Returns (t, X/N, Y/N)."""
    rng = np.random.default_rng(sd)
    ts, Xs, Ys = [0.0], [0], [0]
    t, nX, nY = 0.0, 0, 0
    while t < T:
        a0 = k0 * N
        a1 = (k1 / N) * nX * (nX - 1) / 2 if nX >= 2 else 0.0
        a2 = k2 * nX
        a3 = k3 * nY
        a_tot = a0 + a1 + a2 + a3
        if a_tot <= 0:
            break
        t += rng.exponential(1.0 / a_tot)
        if t > T:
            break
        r = rng.random() * a_tot
        if   r < a0:           nX += 1
        elif r < a0 + a1:      nX -= 2; nY += 1
        elif r < a0 + a1 + a2: nX -= 1
        else:                  nY -= 1
        ts.append(t); Xs.append(nX); Ys.append(nY)
    ts.append(T); Xs.append(nX); Ys.append(nY)
    return np.asarray(ts), np.asarray(Xs, dtype=float)/N, np.asarray(Ys, dtype=float)/N

# Subsample onto a fixed grid for efficient plotting
t_plot = np.linspace(0, T, 3000)

def to_grid(ts, vals):
    idx = np.searchsorted(ts, t_plot, side="right") - 1
    idx = np.clip(idx, 0, len(vals) - 1)
    return vals[idx]

# ── Run SSA for several volumes ───────────────────────────────
volumes = [1, 10, 100, 500]
colors  = ["#0072BD", "#D95319", "#77AC30", "#7E2F8E"]   # MATLAB
alphas  = [0.70, 0.75, 0.80, 0.90]
lws     = [0.8, 0.9, 1.1, 1.4]

ssa_data = {}
for N in volumes:
    print(f"  N = {N:>4d} ...", end=" ", flush=True)
    ts, xs, ys = gillespie_N(T, N, seed)
    xg = to_grid(ts, xs)
    yg = to_grid(ts, ys)
    ssa_data[N] = (xg, yg)
    print(f"{len(ts)-2:>8,d} events,  x(T)/N = {xs[-1]:.2f}")

# ── Figure ────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
fig.subplots_adjust(left=0.10, right=0.96, top=0.88, bottom=0.09,
                    hspace=0.12)

fig.suptitle(
    r"Kurtz's theorem:  $X^{\,N}\!(t) = X(t)/N \;\longrightarrow\; x(t)$"
    r"  as  $N \to \infty$",
    fontsize=14, fontweight="bold")

# ---- X panel ----
for N, c, a, lw in zip(volumes, colors, alphas, lws):
    xg, _ = ssa_data[N]
    ax1.plot(t_plot, xg, color=c, alpha=a, lw=lw,
             label=rf"$N = {N}$")

ax1.plot(tg, x_det, "k--", lw=2.2,
         label=r"Mean-field limit $x(t)$")

ax1.set_ylabel(r"Concentration $x^{\,N}\!(t) = X(t)/N$")
ax1.legend(loc="lower right", framealpha=0.95, edgecolor="0.82",
           ncol=3)
ax1.grid(True, alpha=0.2)
ax1.set_title(
    r"$\varnothing \!\to\! X,\;"
    r"2X \!\to\! Y,\;"
    r"X \!\to\! \varnothing,\;"
    r"Y \!\to\! \varnothing$",
    fontsize=10, pad=4)

# ---- Y panel ----
for N, c, a, lw in zip(volumes, colors, alphas, lws):
    _, yg = ssa_data[N]
    ax2.plot(t_plot, yg, color=c, alpha=a, lw=lw)

ax2.plot(tg, y_det, "k--", lw=2.2)

ax2.set_ylabel(r"Concentration $y^{\,N}\!(t) = Y(t)/N$")
ax2.set_xlabel(r"Time $t$")
ax2.grid(True, alpha=0.2)

for ax in (ax1, ax2):
    ax.set_xlim(0, T)

fig.savefig("kurtz_convergence.svg", bbox_inches="tight")
fig.savefig("kurtz_convergence.png", dpi=200, bbox_inches="tight")
print("\nFigure saved.")
plt.close(fig)